In [ ]:
%pip install selenium beautifulsoup4 pandas webdriver-manager
%pip install webdriver-manager
%pip install selenium
%pip install streamlit
%pip install pandas
%pip install pyperclip
%pip install pymysql

# 주소 구체적으로 사용함 

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import pymysql
import pyperclip

# 1. 네이버 지도 접근
options = webdriver.ChromeOptions()
# options.add_argument("--headless")  # 백그라운드 실행
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.get("https://map.naver.com/p?c=15.00,0,0,0,dh")

# driver = webdriver.Chrome()
# driver.get("https://map.naver.com/p?c=15.00,0,0,0,dh")
search_box = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR,"input.input_search"))
)
print(type(search_box),search_box)

# 2. DB 연결 및 목록 가져오기 (먼저 수행)
conn = pymysql.connect(
    host='127.0.0.1',
    port=3306,
    user='vosnuevo',
    password='vosnuevo',
    database='parking_db',
    charset='utf8mb4' # utf-8이 아니라 utf8mb4를 주로 사용합니다
)

cursor = conn.cursor()
cursor.execute("SELECT pk_code, pk_name, pk_address FROM parking_lots")
rows = cursor.fetchall()
parking_list = [{"pk_code": row[0], "pk_name": row[1], "pk_address": row[2]} for row in rows]
cursor.close()
conn.close()

print("수집할 주차장 목록:")
for p in parking_list:
    print(f"  [{p['pk_code']}] {p['pk_name']} / {p['pk_address']}")
    
# 3. 네이버 지도 크롤링 함수
def clear_search_box(driver, search_box):
    search_box.click()
    search_box.send_keys(Keys.CONTROL + "a")
    search_box.send_keys(Keys.DELETE)
    time.sleep(0.3)
    
    current_value = search_box.get_attribute("value")
    if current_value:
        driver.execute_script("""
            var input = arguments[0];
            var nativeInputValueSetter = Object.getOwnPropertyDescriptor(
                window.HTMLInputElement.prototype, 'value').set;
            nativeInputValueSetter.call(input, '');
            input.dispatchEvent(new Event('input', { bubbles: true }));
        """, search_box)
        time.sleep(0.3)

    print(f"  검색창 초기화: '{search_box.get_attribute('value')}'")


# 4. 검색 + 리뷰 수집 함수
def search_and_get_reviews(driver, pk_code, parking_name, parking_address):
    wait = WebDriverWait(driver, 10)
    results = []

    try:
        # [STEP 1] 항상 메인 컨텍스트로 초기화
        driver.switch_to.default_content()

        # [STEP 2] 검색창에 주차장 이름 입력
        search_box = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "input.input_search"))
        )
        clear_search_box(driver, search_box)            
        search_box.send_keys(parking_name)
        time.sleep(0.3)
        
        # [STEP 3] 입력값 검증
        input_value = search_box.get_attribute("value")
        if input_value != parking_name:
            print(f" 입력 불일치 → 재시도 (현재: '{input_value}')")
            clear_search_box(driver, search_box)
            search_box.send_keys(parking_name)
            time.sleep(0.3)

        search_box.send_keys(Keys.ENTER)
        time.sleep(2)
        
        # [STEP 4] searchIframe 전환
        driver.switch_to.frame("searchIframe")
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "li.VLTHu")))  # ← 이 줄만 추가
        result_items = driver.find_elements(By.CSS_SELECTOR, "li.VLTHu")
                
        # [STEP 5] 이름 일치하는 항목만 순서대로 추출
        candidate_links = []
        for item in result_items:
            try:
                name_text = item.find_element(By.CSS_SELECTOR, "span.YwYLL").text.strip()
                link_el = item.find_element(By.CSS_SELECTOR, "a.U70Fj")
                if name_text == parking_name.strip():
                    candidate_links.append(link_el)
                    print(f"이름 일치 후보 발견: [{name_text}]")
            except:
                continue

        # 이름 일치 없으면 첫 번째 결과 사용
        if not candidate_links:
            print(f"[{parking_name}] 이름 일치 없음 → 첫 번째 결과 사용")
            candidate_links = [result_items[0].find_element(By.CSS_SELECTOR, "a.U70Fj")]  
            
        # [STEP 6] 후보 순서대로 클릭 → entryIframe에서 pz7wy 주소 검증
        target_confirmed = False
        
        for idx, link in enumerate(candidate_links):
            print(f"[{parking_name}] {idx+1}번째 후보 클릭")

            # searchIframe에서 클릭
            driver.switch_to.default_content()
            driver.switch_to.frame("searchIframe")
            driver.execute_script("arguments[0].click();", link)
            time.sleep(3)
            
            # entryIframe 진입 → pz7wy 주소 확인
            driver.switch_to.default_content()
            driver.switch_to.frame("entryIframe")

            try:
                # pz7wy 주소 모두 가져오기 (도로명, 지번 둘 다)
                pz7wy_els = driver.find_elements(By.CSS_SELECTOR, "span.pz7wy")
                naver_addresses = [el.text.strip().rstrip(',').strip() for el in pz7wy_els if el.text.strip()]
                print(f"네이버 주소(pz7wy): {naver_addresses}")
            except:
                naver_addresses = []
                
            # DB 주소와 숫자 포함 비교
            def addr_contains_numbers(db_addr, naver_addr):
                import re
                db_numbers = re.findall(r'\d+', db_addr)
                naver_numbers = re.findall(r'\d+', naver_addr)
                if not db_numbers:
                    return True
                return all(n in naver_numbers for n in db_numbers)

            matched = any(
                (parking_address.strip() in na or na in parking_address.strip()) and
                addr_contains_numbers(parking_address, na)
                for na in naver_addresses
            )

            if matched:
                print(f"[{parking_name}] ✅ pz7wy 주소 일치!")
                target_confirmed = True
                break
            else:
                print(f"[{parking_name}] ❌ 주소 불일치 → 다음 후보")

        if not target_confirmed:
            print(f"[{parking_name}] ⚠️ 주소 일치 없음 → 마지막 클릭한 결과로 진행")
                                        
        # [STEP 7] entryIframe 전환
        driver.switch_to.default_content()
        driver.switch_to.frame("entryIframe")
        
        # [STEP 8] 리뷰 탭 클릭
        try:
            # 모든 탭 목록 출력 (디버깅용)
            all_tabs = driver.find_elements(By.CSS_SELECTOR, "a.tpj9w span.I2hj8")
            tab_texts = [t.text for t in all_tabs]
            print(f"[{parking_name}] 탭 목록: {tab_texts}")
            
            # 클릭 후 리뷰 탭이 선택됐는지 검증
            if "리뷰" not in tab_texts:
                print(f"[{parking_name}] 리뷰 탭 클릭했는데 선택 안 됨 → 스킵")
                return results
            
            # span 텍스트가 정확히 '리뷰'인 것의 부모 a 태그 찾기
            review_span = None
            for span in all_tabs:
                if span.text.strip() == "리뷰":
                    review_span = span
                    break
                
            if review_span is None:
                print(f"[{parking_name}] 리뷰 span 못 찾음 -> 스킵")
                return results
            
            # span의 부모 a 태그 가져오기
            review_tab = driver.execute_script(
                "return arguments[0].closest('a.tpj9w');", review_span
            )
            print(f"[{parking_name}] 리뷰 탭 href: {review_tab.get_attribute('href')}")
            
            # 클릭
            driver.execute_script("arguments[0].click();", review_tab)
            time.sleep(2)
            
            # 검증 : 리뷰 목록이 실제로 로딩되었는지 확인 (aria-selected 대신)
            try:
                wait.until(EC.presence_of_element_located((By.ID, "_review_list")))
                print(f"[{parking_name}] 리뷰 목록 로딩 완료")
            except:
                print(f"[{parking_name}] 리뷰 목록 로딩 실패 → 스킵")
                return results
        except Exception as e:
            print(f"[{parking_name}] 리뷰 탭 오류: {e} → 스킵")
            return results

        # [STEP 9] 더보기 반복하며 리뷰 수집
        collected = set()  # 중복 방지
        
        while True:
            try:
                review_list = driver.find_element(By.ID, "_review_list")
            except:
                print(f"[{parking_name}] _review_list 못 찾음 → 종료")
                break
                
            cards = review_list.find_elements(By.CSS_SELECTOR, "li.EjjAW")
            
            if not cards:
                print(f"[{parking_name}] 카드 없음 → 종료")
                break
            
            for card in cards:
                try:
                    comment = card.find_element(
                        By.CSS_SELECTOR, "a[data-pui-click-code='rvshowmore']"
                        ).text.strip()
                except:
                    comment = "없음"
                    
                try:
                    # 날짜: time 태그
                    date = card.find_element(By.CSS_SELECTOR, "time[aria-hidden='true']").text.strip()
                except:
                    date = "없음"

                if comment != "없음":
                    results.append({
                        "pk_code": pk_code,
                        "pk_name": parking_name,
                        "댓글": comment,
                        "날짜" : date
                    })

            try:
                more_btn = driver.find_element(
                    By.XPATH, "//a[contains(text(),'더보기') or contains(text(),'리뷰 더보기')]"
                )
                driver.execute_script("arguments[0].click();", more_btn)
                wait.until(EC.presence_of_element_located((By.ID, "_review_list")))
            except:
                break

        print(f"[{parking_name}] {len(results)}개 리뷰 수집 완료")

    except Exception as e:
        print(f"[{parking_name}] 오류: {e}")

    finally:
        driver.switch_to.default_content()

    return results
            

# 5. 메인 실행
all_results = []

for parking in parking_list:
    print(f"\n검색 중: [{parking['pk_code']}] {parking['pk_name']} / {parking['pk_address']}")
    reviews = search_and_get_reviews(
        driver,
        parking['pk_code'],
        parking['pk_name'],
        parking['pk_address']
    )
    all_results.extend(reviews)
    time.sleep(1)

driver.quit()

# 6. CSV 저장
df = pd.DataFrame(all_results)
df.to_csv("parking_reviews.csv", index=False, encoding="utf-8")
print(f"\n✅ 완료! 총 {len(all_results)}개 리뷰 저장됨")
print(df.head())

<class 'selenium.webdriver.remote.webelement.WebElement'> <selenium.webdriver.remote.webelement.WebElement (session="89e407531e5dab0e66a05f392d2ec76b", element="f.EBC3899E4D36A4C2FAD91D2280AF3F11.d.169631BE20BD1FD3732BE196DDEA587A.e.3")>
수집할 주차장 목록:
  [1] 강남공영주차장 / 서울 관악구 신림동 1675-8
  [2] 사당역 공영 주차장 / 서울 서초구 과천대로 950-18
  [3] 잠실역 공영 주차장 / 서울 송파구 올림픽로 300
  [4] 마포공영주차장 / 서울 마포구 숭문길 72 소금나루도서관
  [5] 종묘공영주차장 / 서울특별시 종로구 종로 157 1-2

검색 중: [1] 강남공영주차장 / 서울 관악구 신림동 1675-8
  검색창 초기화: ''
이름 일치 후보 발견: [강남공영주차장]
[강남공영주차장] 1번째 후보 클릭
네이버 주소(pz7wy): ['서울 관악구 신림동 1675-8', '영업시간 수정 제안하기']
[강남공영주차장] ✅ pz7wy 주소 일치!
[강남공영주차장] 탭 목록: ['홈', '리뷰', '사진', '정보']
[강남공영주차장] 리뷰 탭 href: https://pcmap.place.naver.com/place/18723890/review?bk_query=%EA%B0%95%EB%82%A8%EA%B3%B5%EC%98%81%EC%A3%BC%EC%B0%A8%EC%9E%A5&entry=bmp&fromPanelNum=2&timestamp=202603170055&locale=ko&svcName=map_pcv5&searchText=%EA%B0%95%EB%82%A8%EA%B3%B5%EC%98%81%EC%A3%BC%EC%B0%A8%EC%9E%A5
[강남공영주차장] 리뷰 목록 로딩 완료
[강남공영주차장] 5개 리뷰 수집 완료

검색 중: [2] 사당역